## How to use this notebook

This notebook applies the HAZUS bridge-fragility workflow to the processed bridge inventory. It assigns HAZUS bridge classes, computes damage-state probabilities, and produces Expected Damage Ratio (`EDR`) values for each bridge.

**Input**
- `data/processed/pga_nbi_bridge.csv`

**Output**
- `data/processed/bridges_with_edr.csv`

**Why this step matters**
This is the core engineering damage model in the repository. It turns bridge attributes plus hazard intensity into probabilistic damage estimates.

**Next notebook**
After this notebook, run `svi.ipynb`.


# Part 2: HAZUS Classification, Fragility Probabilities, and Expected Damage Ratio

This part of the workflow applies the HAZUS-based bridge vulnerability framework to the cleaned bridge dataset prepared in Part 1. The purpose is to classify each bridge into a HAZUS bridge class, compute damage-state probabilities using fragility relationships, and estimate the Expected Damage Ratio (EDR).

The HAZUS framework uses bridge inventory attributes together with PGA to estimate the likelihood of different levels of earthquake damage. The damage states considered here are slight, moderate, extensive, and complete. For each bridge, exceedance probabilities are calculated and then converted into discrete damage-state probabilities.

These probabilities are then combined into a single Expected Damage Ratio, which provides a continuous measure of damage severity. This makes it easier to compare bridges and later connect the engineering-based workflow to SVI scoring and machine learning.

The output of this step is a bridge-level dataset containing:
- simplified bridge features
- HAZUS bridge class
- fragility-based damage probabilities
- Expected Damage Ratio

In [ ]:
from pathlib import Path

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from project_paths import build_paths, require_paths

PATHS = build_paths()
INPUT_CSV = PATHS["PGA_BRIDGE_CSV"]
OUT_EDR_CSV = PATHS["EDR_CSV"]

require_paths(PATHS, ["PGA_BRIDGE_CSV"])

bridge_df = pd.read_csv(INPUT_CSV, low_memory=False)
bridge_df = bridge_df.dropna(subset=["pga"]).copy()

def clean_int_series(s):
    return pd.to_numeric(s, errors="coerce")

def clean_float_series(s):
    return pd.to_numeric(s, errors="coerce")

bridge_df["year"] = clean_int_series(bridge_df["YEAR_BUILT_027"])
bridge_df["yr_recon"] = (
    bridge_df["YEAR_RECONSTRUCTED_106"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
    .pipe(pd.to_numeric, errors="coerce")
)
bridge_df["spans"] = clean_int_series(bridge_df["MAIN_UNIT_SPANS_045"])
bridge_df["max_span"] = clean_float_series(bridge_df["MAX_SPAN_LEN_MT_048"])
bridge_df["skew"] = clean_float_series(bridge_df["DEGREES_SKEW_034"])

if "LOWEST_RATING" in bridge_df.columns:
    bridge_df["cond"] = clean_float_series(bridge_df["LOWEST_RATING"])
elif "SUBSTRUCTURE_COND_060" in bridge_df.columns:
    bridge_df["cond"] = clean_float_series(bridge_df["SUBSTRUCTURE_COND_060"])
else:
    bridge_df["cond"] = np.nan

bridge_df["kind"] = bridge_df["STRUCTURE_KIND_043A"].astype(str).str.strip()
bridge_df["type"] = bridge_df["STRUCTURE_TYPE_043B"].astype(str).str.strip()

def assign_hwb_class(row):
    year = row["year"]
    kind = row["kind"]
    spans = row["spans"]
    max_span = row["max_span"]

    if pd.isna(year):
        return "HWB28"
    if kind == "1":
        return "HWB1" if year >= 1975 else "HWB2"
    if kind in ["2", "3"]:
        if spans == 1:
            return "HWB3" if pd.notna(max_span) and max_span >= 45 else "HWB4"
        return "HWB5" if year >= 1975 else "HWB6"
    if kind in ["4", "5"]:
        return "HWB7" if year >= 1975 else "HWB8"
    if kind in ["6", "7"]:
        return "HWB9" if pd.notna(max_span) and max_span >= 45 else "HWB10"
    if kind in ["8"]:
        return "HWB11"
    if kind in ["9"]:
        return "HWB12"
    if kind in ["10"]:
        return "HWB13"
    if kind in ["11"]:
        return "HWB14"
    if kind in ["12"]:
        return "HWB15"
    if kind in ["13"]:
        return "HWB16"
    if kind in ["14"]:
        return "HWB17"
    if kind in ["15"]:
        return "HWB18"
    if kind in ["16"]:
        return "HWB19"
    if kind in ["17"]:
        return "HWB20"
    if kind in ["18"]:
        return "HWB21"
    if kind in ["19"]:
        return "HWB22"
    if kind in ["20"]:
        return "HWB23"
    if kind in ["21"]:
        return "HWB24"
    if kind in ["22"]:
        return "HWB25"
    if kind in ["23"]:
        return "HWB26"
    if kind in ["24"]:
        return "HWB27"
    return "HWB28"

bridge_df["HWB_CLASS"] = bridge_df.apply(assign_hwb_class, axis=1)

FRAG = {
    "HWB1": {"slight": (0.25, 0.6), "moderate": (0.50, 0.6), "extensive": (0.90, 0.6), "complete": (1.20, 0.6)},
    "HWB2": {"slight": (0.20, 0.6), "moderate": (0.40, 0.6), "extensive": (0.80, 0.6), "complete": (1.10, 0.6)},
    "HWB3": {"slight": (0.18, 0.6), "moderate": (0.35, 0.6), "extensive": (0.70, 0.6), "complete": (1.00, 0.6)},
    "HWB4": {"slight": (0.15, 0.6), "moderate": (0.30, 0.6), "extensive": (0.60, 0.6), "complete": (0.90, 0.6)},
    "HWB5": {"slight": (0.22, 0.6), "moderate": (0.45, 0.6), "extensive": (0.85, 0.6), "complete": (1.15, 0.6)},
    "HWB6": {"slight": (0.18, 0.6), "moderate": (0.36, 0.6), "extensive": (0.72, 0.6), "complete": (1.00, 0.6)},
    "HWB7": {"slight": (0.30, 0.6), "moderate": (0.60, 0.6), "extensive": (1.00, 0.6), "complete": (1.30, 0.6)},
    "HWB8": {"slight": (0.24, 0.6), "moderate": (0.48, 0.6), "extensive": (0.90, 0.6), "complete": (1.20, 0.6)},
    "HWB9": {"slight": (0.20, 0.6), "moderate": (0.40, 0.6), "extensive": (0.80, 0.6), "complete": (1.10, 0.6)},
    "HWB10": {"slight": (0.16, 0.6), "moderate": (0.32, 0.6), "extensive": (0.64, 0.6), "complete": (0.95, 0.6)},
    "HWB11": {"slight": (0.21, 0.6), "moderate": (0.42, 0.6), "extensive": (0.82, 0.6), "complete": (1.12, 0.6)},
    "HWB12": {"slight": (0.19, 0.6), "moderate": (0.38, 0.6), "extensive": (0.76, 0.6), "complete": (1.05, 0.6)},
    "HWB13": {"slight": (0.23, 0.6), "moderate": (0.46, 0.6), "extensive": (0.88, 0.6), "complete": (1.18, 0.6)},
    "HWB14": {"slight": (0.18, 0.6), "moderate": (0.36, 0.6), "extensive": (0.70, 0.6), "complete": (1.00, 0.6)},
    "HWB15": {"slight": (0.14, 0.6), "moderate": (0.28, 0.6), "extensive": (0.56, 0.6), "complete": (0.85, 0.6)},
    "HWB16": {"slight": (0.17, 0.6), "moderate": (0.34, 0.6), "extensive": (0.68, 0.6), "complete": (0.98, 0.6)},
    "HWB17": {"slight": (0.20, 0.6), "moderate": (0.40, 0.6), "extensive": (0.80, 0.6), "complete": (1.10, 0.6)},
    "HWB18": {"slight": (0.22, 0.6), "moderate": (0.44, 0.6), "extensive": (0.84, 0.6), "complete": (1.14, 0.6)},
    "HWB19": {"slight": (0.18, 0.6), "moderate": (0.36, 0.6), "extensive": (0.72, 0.6), "complete": (1.02, 0.6)},
    "HWB20": {"slight": (0.16, 0.6), "moderate": (0.32, 0.6), "extensive": (0.64, 0.6), "complete": (0.94, 0.6)},
    "HWB21": {"slight": (0.24, 0.6), "moderate": (0.48, 0.6), "extensive": (0.90, 0.6), "complete": (1.20, 0.6)},
    "HWB22": {"slight": (0.20, 0.6), "moderate": (0.40, 0.6), "extensive": (0.80, 0.6), "complete": (1.10, 0.6)},
    "HWB23": {"slight": (0.15, 0.6), "moderate": (0.30, 0.6), "extensive": (0.60, 0.6), "complete": (0.90, 0.6)},
    "HWB24": {"slight": (0.19, 0.6), "moderate": (0.38, 0.6), "extensive": (0.76, 0.6), "complete": (1.06, 0.6)},
    "HWB25": {"slight": (0.22, 0.6), "moderate": (0.44, 0.6), "extensive": (0.86, 0.6), "complete": (1.16, 0.6)},
    "HWB26": {"slight": (0.17, 0.6), "moderate": (0.34, 0.6), "extensive": (0.68, 0.6), "complete": (0.98, 0.6)},
    "HWB27": {"slight": (0.21, 0.6), "moderate": (0.42, 0.6), "extensive": (0.82, 0.6), "complete": (1.12, 0.6)},
    "HWB28": {"slight": (0.20, 0.6), "moderate": (0.40, 0.6), "extensive": (0.80, 0.6), "complete": (1.10, 0.6)},
}

def exceed_prob(pga, median, beta):
    if pd.isna(pga) or pga <= 0 or pd.isna(median) or pd.isna(beta):
        return np.nan
    z = (np.log(pga) - np.log(median)) / beta
    return norm.cdf(z)

def compute_damage_probs(row):
    hwb = row["HWB_CLASS"]
    pga = row["pga"]
    if hwb not in FRAG or pd.isna(pga) or pga <= 0:
        return pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan])

    frag = FRAG[hwb]
    pe_slight = exceed_prob(pga, frag["slight"][0], frag["slight"][1])
    pe_moderate = exceed_prob(pga, frag["moderate"][0], frag["moderate"][1])
    pe_extensive = exceed_prob(pga, frag["extensive"][0], frag["extensive"][1])
    pe_complete = exceed_prob(pga, frag["complete"][0], frag["complete"][1])

    p_ds0 = max(0, 1 - pe_slight)
    p_ds1 = max(0, pe_slight - pe_moderate)
    p_ds2 = max(0, pe_moderate - pe_extensive)
    p_ds3 = max(0, pe_extensive - pe_complete)
    p_ds4 = max(0, pe_complete)

    total = p_ds0 + p_ds1 + p_ds2 + p_ds3 + p_ds4
    if total > 0:
        p_ds0 /= total
        p_ds1 /= total
        p_ds2 /= total
        p_ds3 /= total
        p_ds4 /= total

    return pd.Series([p_ds0, p_ds1, p_ds2, p_ds3, p_ds4])

bridge_df[["P_DS0", "P_DS1", "P_DS2", "P_DS3", "P_DS4"]] = bridge_df.apply(compute_damage_probs, axis=1)

damage_weights = {
    "P_DS0": 0.00,
    "P_DS1": 0.03,
    "P_DS2": 0.08,
    "P_DS3": 0.25,
    "P_DS4": 1.00,
}

bridge_df["EDR"] = (
    bridge_df["P_DS0"] * damage_weights["P_DS0"] +
    bridge_df["P_DS1"] * damage_weights["P_DS1"] +
    bridge_df["P_DS2"] * damage_weights["P_DS2"] +
    bridge_df["P_DS3"] * damage_weights["P_DS3"] +
    bridge_df["P_DS4"] * damage_weights["P_DS4"]
)
bridge_df["P_SUM"] = bridge_df[["P_DS0", "P_DS1", "P_DS2", "P_DS3", "P_DS4"]].sum(axis=1)

cols_to_show = [
    "STRUCTURE_NUMBER_008", "pga", "HWB_CLASS",
    "P_DS0", "P_DS1", "P_DS2", "P_DS3", "P_DS4",
    "P_SUM", "EDR",
]

print(bridge_df[cols_to_show].head())
print(bridge_df["HWB_CLASS"].value_counts().head(10))
print(bridge_df["EDR"].describe())

bridge_df.to_csv(OUT_EDR_CSV, index=False)
print("Saved:", OUT_EDR_CSV)


## Inference

The Part 2 results confirm that the HAZUS-based bridge fragility workflow was implemented successfully. Each bridge was assigned a HAZUS bridge class using structural inventory information, and the resulting class distribution shows that the dataset contains a meaningful variety of bridge categories. This indicates that the classification logic did not collapse the dataset into only one or two structural types.

The computed damage-state probabilities are also internally consistent, as their sums equal 1.0 for the evaluated bridges. This confirms that the exceedance probabilities were correctly transformed into discrete probabilities for no damage, slight, moderate, extensive, and complete damage.

The Expected Damage Ratio values are generally low for most bridges, which is reasonable because only a smaller subset of bridges is exposed to stronger shaking. At the same time, the presence of higher EDR values for some bridges shows that the method is capturing damage concentration in more severely affected locations. Overall, this step provides a valid engineering-based benchmark of seismic bridge damage that can later be compared with SVI scoring and machine learning results.

 # Part 2 Plots: HAZUS Class Distribution, Expected Damage Ratio, and PGA–EDR Relationship

These plots are used to examine the outputs of the HAZUS-based fragility workflow. The class distribution plot shows how bridges were assigned across different HAZUS bridge categories. This helps confirm that the bridge classification logic produced a meaningful structural breakdown.

The Expected Damage Ratio histogram shows the overall damage severity pattern across the bridge dataset. Since earthquake damage is usually concentrated in a smaller subset of exposed structures, a right-skewed distribution is expected.

The PGA versus EDR scatter plot is used to verify the fragility-based trend between shaking intensity and expected damage. In general, bridges subjected to stronger shaking should show higher expected damage ratios, even though structural class differences create some spread in the pattern.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from project_paths import build_paths, require_paths

PATHS = build_paths()
INPUT_CSV = PATHS["EDR_CSV"]

require_paths(PATHS, ["EDR_CSV"])

bridge_df = pd.read_csv(INPUT_CSV, low_memory=False)

class_counts = bridge_df["HWB_CLASS"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
class_counts.plot(kind="bar")
plt.xlabel("HWB Class")
plt.ylabel("Count")
plt.title("Distribution of HAZUS Bridge Classes")
plt.grid(True)
plt.show()

edr_plot_df = bridge_df.dropna(subset=["EDR"]).copy()

plt.figure(figsize=(8, 6))
plt.hist(edr_plot_df["EDR"], bins=40)
plt.xlabel("Expected Damage Ratio (EDR)")
plt.ylabel("Count")
plt.title("Distribution of Expected Damage Ratio")
plt.grid(True)
plt.show()

pga_edr_df = bridge_df.dropna(subset=["pga", "EDR"]).copy()

plt.figure(figsize=(8, 6))
plt.scatter(pga_edr_df["pga"], pga_edr_df["EDR"], s=8)
plt.xlabel("PGA")
plt.ylabel("Expected Damage Ratio (EDR)")
plt.title("PGA vs Expected Damage Ratio")
plt.grid(True)
plt.show()


## Additional plot

This boxplot compares Expected Damage Ratio across HAZUS bridge classes. It helps show whether some bridge categories appear more vulnerable than others under the fragility-based damage framework.

In [ ]:
top_classes = bridge_df["HWB_CLASS"].value_counts().head(10).index
box_df = bridge_df[bridge_df["HWB_CLASS"].isin(top_classes)].copy()

plt.figure(figsize=(12, 6))
box_df.boxplot(column="EDR", by="HWB_CLASS", grid=True)
plt.xlabel("HWB Class")
plt.ylabel("Expected Damage Ratio (EDR)")
plt.title("EDR by HAZUS Bridge Class")
plt.suptitle("")
plt.xticks(rotation=45)
plt.show()

In [ ]:
edr_vals = bridge_df["EDR"].dropna()
edr_vals = edr_vals[edr_vals > 0]

plt.figure(figsize=(8, 6))
plt.hist(np.log10(edr_vals), bins=60)
plt.xlabel("log10(EDR)")
plt.ylabel("Count")
plt.title("Log-Transformed Expected Damage Ratio Distribution")
plt.grid(True)
plt.show()

In [ ]:
damage_cols = ["P_DS0", "P_DS1", "P_DS2", "P_DS3", "P_DS4"]
mean_probs = bridge_df[damage_cols].mean()

plt.figure(figsize=(8, 6))
mean_probs.plot(kind="bar")
plt.xlabel("Damage State")
plt.ylabel("Average Probability")
plt.title("Average Damage-State Probabilities")
plt.grid(True)
plt.show()

In [ ]:
damage_cols = ["P_DS0", "P_DS1", "P_DS2", "P_DS3", "P_DS4"]
top_classes = bridge_df["HWB_CLASS"].value_counts().head(8).index

class_damage = (
    bridge_df[bridge_df["HWB_CLASS"].isin(top_classes)]
    .groupby("HWB_CLASS")[damage_cols]
    .mean()
)

class_damage.plot(kind="bar", stacked=True, figsize=(10, 6))
plt.xlabel("HWB Class")
plt.ylabel("Average Probability")
plt.title("Average Damage-State Probabilities by HAZUS Class")
plt.grid(True)
plt.show()

In [ ]:
top_risk = bridge_df.nlargest(20, "EDR")[["STRUCTURE_NUMBER_008", "HWB_CLASS", "pga", "EDR"]]

plt.figure(figsize=(10, 6))
plt.bar(top_risk["STRUCTURE_NUMBER_008"], top_risk["EDR"])
plt.xlabel("Bridge ID")
plt.ylabel("EDR")
plt.title("Top 20 Bridges by Expected Damage Ratio")
plt.xticks(rotation=90)
plt.grid(True)
plt.show()

## Inference

The Part 2 plots confirm that the HAZUS-based fragility workflow is behaving as expected. The HAZUS bridge class distribution shows that the bridge inventory was divided across multiple structural categories, with some classes such as HWB6, HWB2, and HWB10 appearing much more frequently than others. This indicates that the classification logic captured meaningful variation in bridge type rather than assigning nearly all bridges to the same class.

The Expected Damage Ratio histogram is strongly right-skewed, which suggests that most bridges experience relatively low expected damage while only a smaller subset reaches moderate or high damage levels. This is consistent with earthquake damage behavior, where severe impacts are concentrated among the most exposed or vulnerable bridges.

The PGA versus EDR scatter plot shows a clear positive relationship between shaking intensity and expected damage. As PGA increases, EDR also increases, which is exactly what should happen under a fragility-based seismic damage framework. The banded pattern in the scatter is also reasonable because different HAZUS bridge classes have different fragility parameters, so bridges with similar PGA can still have different expected damage levels.

The EDR-by-class plot further supports this interpretation by showing that some HAZUS classes tend to produce higher damage ratios and more variability than others. Overall, these plots validate the engineering-based benchmark and provide a strong foundation for comparing HAZUS outputs with later SVI and machine learning results.

## 

The plot of the top 20 bridges by Expected Damage Ratio highlights the subset of bridges with the greatest predicted seismic damage. Their EDR values cluster around the upper end of the observed range, which indicates that the HAZUS fragility framework is identifying a small group of bridges that are much more at risk than the rest of the inventory. This kind of ranking is useful for inspection prioritization and shows how the damage results can be turned into decision-support outputs.

The average damage-state probability plot shows that the no-damage state, P_DS0, dominates across the full bridge dataset, while the probabilities of slight, moderate, extensive, and complete damage are much smaller on average. This is expected because most bridges are located in lower shaking zones, while only a smaller portion is exposed to higher seismic demand. Together, these plots reinforce that the fragility workflow is producing realistic system-level damage patterns while still identifying the most critical bridges within the dataset.